In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/gme_database.db')
print("Database created!")

Database created!


In [10]:
#cleaned data
stock = pd.read_csv(r"C:\Users\praful\Desktop\madhulika\sentiment stock analysis\data\gme_stock_clean.csv")
reddit = pd.read_csv(r"C:\Users\praful\Desktop\madhulika\sentiment stock analysis\data\gme_reddit_clean.csv")

stock.to_sql('stock_prices', conn, if_exists='replace', index=False)
reddit.to_sql('reddit_posts', conn, if_exists='replace', index=False)

print("Tables created successfully!")
print("Tables in database:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist())

Tables created successfully!
Tables in database: ['stock_prices', 'reddit_posts']


In [11]:
#stock table
stock_check = pd.read_sql("SELECT * FROM stock_prices LIMIT 5", conn)
print("STOCK TABLE:")
print(stock_check)

print()

#reddit table
reddit_check = pd.read_sql("SELECT * FROM reddit_posts LIMIT 5", conn)
print("REDDIT TABLE:")
print(reddit_check)

STOCK TABLE:
         Date   Close    High     Low    Open    Volume
0  2021-01-04  4.3125  4.7750  4.2875  4.7500  40090000
1  2021-01-05  4.3425  4.5200  4.3075  4.3375  19846000
2  2021-01-06  4.5900  4.7450  4.3325  4.3350  24224800
3  2021-01-07  4.5200  4.8625  4.5050  4.6175  24517200
4  2021-01-08  4.4225  4.5750  4.2700  4.5450  25928000

REDDIT TABLE:
       id                                              title  score  \
0  ll0n4p  Need explanations on Level 2 data for GME, why...      1   
1  ll0d5c                                          SAS = GME      1   
2  lkzzpj                 Is anybody still holding onto GME?      1   
3  lkzzc4                      Should I sell GameStop stock?      1   
4  lkzmms  Show is over for GME and AMC? What happened to...      1   

         date  num_comments  
0  2021-02-16             2  
1  2021-02-16             2  
2  2021-02-16             2  
3  2021-02-16             2  
4  2021-02-16             2  


In [12]:
posts_per_day = pd.read_sql_query("""
    SELECT date, COUNT(*) as total_posts
    FROM reddit_posts
    GROUP BY date
    ORDER BY date
""", conn)

print(posts_per_day)

           date  total_posts
0    2012-06-01            1
1    2013-05-09            1
2    2014-11-21            1
3    2014-12-10            1
4    2015-05-29            1
..          ...          ...
492  2021-02-12          555
493  2021-02-13          327
494  2021-02-14          219
495  2021-02-15          218
496  2021-02-16          100

[497 rows x 2 columns]


In [13]:
#Top 5 highest stock price days
top_stock_days = pd.read_sql_query("""
    SELECT Date, Close, Volume
    FROM stock_prices
    ORDER BY Close DESC
    LIMIT 5
""", conn)
print(top_stock_days)

         Date      Close     Volume
0  2021-01-27  86.877502  373586800
1  2021-01-29  81.250000  202264400
2  2021-02-01  56.250000  149528800
3  2021-01-28  48.400002  235263200
4  2021-01-26  36.994999  714352000


In [14]:
combined = pd.read_sql_query("""
    SELECT s.Date, s.Close, s.Volume, r.total_posts
    FROM stock_prices s
    JOIN (
        SELECT date, COUNT(*) as total_posts
        FROM reddit_posts
        GROUP BY date
    ) r
    ON s.Date = r.date
    ORDER BY s.Date
""", conn)
print(combined)

          Date      Close     Volume  total_posts
0   2021-01-04   4.312500   40090000           76
1   2021-01-05   4.342500   19846000           60
2   2021-01-06   4.590000   24224800           49
3   2021-01-07   4.520000   24517200           31
4   2021-01-08   4.422500   25928000           55
5   2021-01-11   4.985000   59632000          106
6   2021-01-12   4.987500   28242800           50
7   2021-01-13   7.850000  578006800          539
8   2021-01-14   9.977500  374869600          630
9   2021-01-15   8.875000  187465600          688
10  2021-01-19   9.840000  298887600          597
11  2021-01-20   9.780000  133887200          463
12  2021-01-21  10.757500  224867600          537
13  2021-01-22  16.252501  788631600         1383
14  2021-01-25  19.197500  711496000         2823
15  2021-01-26  36.994999  714352000         1512
16  2021-01-27  86.877502  373586800         7543
17  2021-01-28  48.400002  235263200        22297
18  2021-01-29  81.250000  202264400        13029


In [16]:
conn.close()
print("Database connection closed successfully!")

Database connection closed successfully!
